# Spark SQL Learning 
##  By knowing the process of Read and Write , become a  Data INGESTION Developer 

connecting various sources (files or filesustem , db , dwh , api ,.. ) and loading data into storage env(data lake)

- csv
- json
- xml
- parquet 
- ORC
- sql
etc....


In [0]:
spark.version

## Few Facts about Unity Catalog

Dataobjects  is managed  in catalog with three namespace 


Catalog -> per domain or environemnt 

schema -> database 

tables , views , functions , volume 

Volume -  Non tabular data (files) , goverened access 

Catalog >> Schema  >> 
                    Table
                    View
                    functions
                    volume 

Volumes are used for managing Non tabulat data (files)

In [0]:
%sql
create catalog if not exists izwd37dev;

create schema if  not exists izwd37dev.wd37db;

create volume if not exists izwd37dev.wd37db.rawdatta;



## DBFS 

### Databricks File system 

distribuited virtual file system , linux posix format  runninng on top of your cloud storages 


/Volumes/catalog/schema/volume-name/path/to/file



dbfs:/ - uri   -> uniform resource identifier  (dbricks file system )

hdfs:/    -> hadoop distruibuited file system 

file:/     -> local file 

s3a:/   -> aws s3

gcs:/  -> google storage


adls:/   -> azure datalake 





In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/"

In [0]:
print(dbutils.fs.ls("dbfs:/Volumes/izwd37dev/wd37db/rawdatta"))

In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv"

In [0]:
%fs head "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs"

In [0]:
%fs mkdirs "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv"

In [0]:
dbutils.fs.ls("dbfs:/Volumes/izwd37dev/wd37db/rawdatta")

In [0]:
dbutils.fs.mkdirs("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json")

what is dbfs:/ -> uri 
        hdfs:/ 
        file:/
        s3:/
        gcs:/

In [0]:
# read data from spark 
# spark SQL 

Spark session 


from pyspark.sql import SparkSession 

spark=SparkSession.builder.getOrCreate()

In [0]:
print(spark)

create Dataframe from storage (files / dir )
To read the delimited data from any storage (dbfs , hdfs , lfs , cloud staorgaes )

spark.read.csv opition  -> dataframe 

-- csv is the built in source 


## Deafults
spark.read.csv 

default options :

header = False 

default cols = _c0 , _c1 _c2

delimiter = ","

default data type for all columns   = string 

In [0]:
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs") # file path
print(type(custdata)) # dataframe
custdata.show() # similar to collect -> action )
# show action , display default 20 records 

In [0]:
# describe table 
custdata.printSchema()

In [0]:
# view few or more records 
custdata.show(10,False)
#custdata.show(200)

In [0]:
display(custdata)

In [0]:
# changing the column names from _c0, _c1 to actuals 
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs").toDF("custid","fname","lname","age","profession")

custdata.show(5) # display only 5 records 

custdata.printSchema()

### supose we recivied a file with header 
### column names we need to pick from the header 

In [0]:
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header")

custdata.show(5)

custdata.printSchema()

In [0]:
# how to get the number of records in the dataframe
# select count(1) from table 
# count is an action , its return integer result , trigger execution , job is created
custdata.count()

overriding the defaults with option 

enable the header 

In [0]:
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True)

custdata.show(5)

custdata.printSchema()
print(f"Number of records in cust : {custdata.count()}")

In [0]:
# data type for all columns treated default as string
# based on the data we have generate the schema with proper data type 
# performance if we read large data inferSchema is not a good option 
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True,inferSchema=True)

custdata.show(5)

custdata.printSchema()


## Different delimited (|)

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp1.csv

In [0]:
emp_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",sep="|",header=True,inferSchema=True)
emp_df.show()

emp_df.printSchema()


spark.read.csv() 

-> required input **path**   

-> path could be a file or dir , list of dir ...

In [0]:
# options , way to create dataframe using csv with different options 
custdata=spark.read.csv(path="dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv") # directory 
custdata.show(5)
custdata.count()

In [0]:
# creating a datfrme from list of paths 
custdata=spark.read.csv(path=["/Volumes/izwd37dev/wd37db/rawdatta/csv/custs","/Volumes/izwd37dev/wd37db/rawdatta/csv/cust2.csv"]) # multiple files  
custdata.show(5)
custdata.count()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai/sales_chennai.csv

In [0]:
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai/sales*",header=True,inferSchema=True)
sales_df.show(1000)
print(sales_df.count())
sales_df.printSchema()

In [0]:
# recursiveFileLookup - read all sub dir
# pathGlobFilter - apply the filter / pattern globally on all dir /sub dir 
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/salesdata",header=True,recursiveFileLookup=True,pathGlobFilter="sales*")
sales_df.show(5)
print(sales_df.count())
sales_df.printSchema()

In [0]:
%fs ls /Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai

In [0]:
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/salesdata",header=True,recursiveFileLookup=True,pathGlobFilter="sales*")

sales_df.count()

In [0]:
# inferSchema -> generating schema by reading the entire data 
# to avoid reading the data for genrating schema , - performance issue 
# when we are going with inferschema its more dynamic  - dq issue 

emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",sep="|",header=True,inferSchema=True)
emp_df.show()
emp_df.printSchema() 

### we can create our own schema and apply while reading data 

In [0]:
# simple way to define schema 
'''
create table cust(custid int,fname string,lname string,age int , prof string)
'''
# creating the schema using ddl string / string 
customer_schema="custid int,fname string,lname string,age int , prof string"

custdata=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/csv/custs",schema=customer_schema) # multiple files  
custdata.show(5)
custdata.count()
custdata.printSchema()

In [0]:
schema_string="eid int,name string,age int"
emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",sep="|",header=True,schema=schema_string)
emp_df.show()
emp_df.printSchema() 

print(emp_df.schema)

In [0]:
# programmatically define the schema - using StructType and StructField

from pyspark.sql.types import StructType,StructField,IntegerType,StringType

'''
int -> integrType
str -> StringType

StructType - structure (row) / record 
StructField - column -> [name , type ]


'''
schema=StructType([
    StructField("eid",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("age",IntegerType(),True)])

emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",sep="|",header=True,schema=schema)
emp_df.show()
emp_df.printSchema() 



## built in Source

- CSV
- JSON
- XML 
- parquet 
- delta
- ORC
- Excel
- jdbc
- table

## Read data from JSON

Javascript object notation 

semi structure data 

key value data 

along with data , it will keep the field name as well

key - colum , value -data 

column ordering not important , number of fields also dynamic 

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/json/emp.json

In [0]:
emp_scehma="id int,name string,age int"
emp_json_df=spark.read.json("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",schema=emp_scehma)

emp_json_df.show()
emp_json_df.printSchema()


In [0]:
# connect external source 
# genric way to read data using spark

# spark.read.option("k","v").format("source").load()

# csv 

# df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs")


cust_df=spark.read.option("header","True").option("inferSchema","True").option("delimiter",",").format("csv").load("/Volumes/izwd37dev/wd37db/rawdatta/custs_header")

cust_df.show()


In [0]:
emp_scehma="id int,name string,age int"
#emp_json_df=spark.read.json("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",schema=emp_scehma)

emp_json_df=spark.read.schema(emp_scehma).format("json").load("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json")

emp_json_df.show()
emp_json_df.printSchema()

In [0]:
# create dataframe from delimited file , comes from any storage 
# local 
# hdfs 
# cloud storages - s3 , gcs , adls
# databricks file system - dbfs

df=spark.read.option("inferSchema","True").csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True)
# or
# df=spark.read.format("csv").load()
df.show(5)
df.printSchema()

In [0]:
# create data frame from json 


cust_df=spark.read.json("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custjson/")
cust_df.show(5)
cust_df.printSchema()
#

In [0]:
# create dataframe from xml


cust_df=spark.read.xml("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custxml/",rowTag="customer")
cust_df.show(5)
cust_df.printSchema()
#

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema)
stud_df.show()
stud_df.printSchema()


# bad record (not matching the schema )
# allow - default 
# fail 
# ignore 

option -> mode 
1. permissive -> permitting , RCA later
2. dropmalformed - blindly drop that record proceed 
3. failfast - immediately fail the whole job

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="PERMISSIVE")
stud_df.show()
stud_df.printSchema()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="dropMalformed")
stud_df.show()
# print(stud_df.count())- 5 record will show actual we have only 3
stud_df.printSchema()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="failFast")
stud_df.show() 
stud_df.printSchema()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int,error_rec string"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="PERMISSIVE",columnNameOfCorruptRecord="error_rec")
stud_df.show()
stud_df.printSchema()

### Built in Source -  csv  -> delimited file coming from any storages (local , hdfs , cloud object storage (s3))

###  semi structure format - json / xml -> 

###  - sturucture / databricks sql / lakehouse table -> spark.read.table() / spark.sql("sql query ")

### Rdbms Source -> spark.read.jdbc() -( cover as part of cloud )

reading a comma seprated employee data  (eid,ename,address)

address colum,n also have comma 

source sending header ifo / footer 

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp3.csv

In [0]:
emp_schema="eid int,ename string,address string"
emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp3.csv",sep=",",header=True,schema=emp_schema,quote="'",escape="~",comment="#")
emp_df.show(10,False) # show default first 20 records 
emp_df.printSchema() 

emp df which has comma in address column 

save emp df in dfs  empout 

In [0]:
emp_df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/empout",header=True,sep=",",quote="'",escape="~")

### create dataframe from python objects 
### programtically create dataframe 

In [0]:

# python list to spark dataframe

data=[(1,"jaya,anu",23),(2,"ram,shyam",25),(3,"sita,geeta",27)]
col_name=["id","name","age"]

df=spark.createDataFrame(data,col_name)
df.show()
df.printSchema()


In [0]:
df.write.csv("/Volumes/izwd37dev/wd37db/rawdatta/pythondfout",header=True,sep=",",quote="'")

In [0]:
e_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/pythondfout",header=True,sep=",",quote="'")
e_df.show()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp2.json

In [0]:
# Json options explore
# yyyy-MM-dd - spark date format 


json_ddl_schema="id int,name string,dob date"
e_json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/emp2.json",multiLine=True,schema=json_ddl_schema,
                        dateFormat="dd/MM/yyyy",allowUnquotedFieldNames=True,allowSingleQuotes=True )
e_json_df.show()

e_json_df.printSchema()

## Schema Evolution 
### changes in the scehma 

cust data 
--------------

Day -1 -> custid ,name , age 
Day 2 ->  custid ,name , age 
Day 3 ->custid ,name , age , profession

Day4 -> custid ,name , age ,profession ,lname 


In [0]:
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo",header=True)

df.show()

In [0]:
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo/day1.csv",header=True)

df.write.mode("overwrite").parquet("/Volumes/izwd37dev/wd37db/rawdatta/schema_out")

In [0]:
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo/day4.csv",header=True)

df.write.mode("append").parquet("/Volumes/izwd37dev/wd37db/rawdatta/schema_out")

In [0]:
# similar thing we are going to implement with CSV 
emp_df=spark.read.parquet("/Volumes/izwd37dev/wd37db/rawdatta/schema_out",mergeSchema=True)
emp_df.show()